In [1]:
import os
import SimpleITK as sitk
from tqdm.notebook import tqdm
import pandas as pd
import numpy as np
import seaborn as sns
from skimage import measure

import radiomics
import torchio as tio
import torch

from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

import matplotlib.pyplot as plt
from ipywidgets import interact, widgets

NUM_WORKERS = 14

AUG_COUNT = 5
AUG_TYPE = "inout_plane"
BIAS = "random"

PARAM_settingsFile = r"/home/thulasiseetha/research/sithin_projects/ACCRadiomics_CT/radiomicsFeatures/radiomicsSettingsCT.yaml"

IN_AUG_PARAMS = {'w_stdMM':0.5, 'h_stdMM':0.5, 'angle':5, 'ob_type':BIAS} #w_stdMM an h_stdMM were selected based on DICE_MIN and HD_MIN
OUT_AUG_PARAMS = {'scale_a':0.6, 'scale_b':0.8, 'angle':5, 'delta_z':2}

DICE_MIN, HD_MIN = 0.75, 5


DATA_DIR = r"/home/thulasiseetha/research/sithin_projects/ACC_Protoni/BASELINE_Nifti/"
ROI = ["new_GTV", "T_RAD"]

XL_PATH = r"/home/thulasiseetha/research/sithin_projects/ACCRadiomics_CT/radiomicsFeatures/features_GTV.xlsx"
SIGNATURES_PATH = r"/home/thulasiseetha/research/sithin_projects/ACCRadiomics_CT/results/selected_signature.npy"


In [2]:
GTV_DF = pd.read_excel(XL_PATH, index_col=0)
PIDS = GTV_DF.pts_id.to_list()
SIGNATURES = np.load(SIGNATURES_PATH, allow_pickle=True).item()

In [3]:

def compute_dice(label, synthetic_label):
    
    label = sitk.Cast(label>0, sitk.sitkUInt8)
    synthetic_label = sitk.Cast(synthetic_label>0, sitk.sitkUInt8)
    
    overlap_measures = sitk.LabelOverlapMeasuresImageFilter()
    overlap_measures.Execute(label, synthetic_label)
    
    dice = overlap_measures.GetDiceCoefficient()
    
    return dice

def compute_hausdorff(label, synthetic_label):
    
    label = sitk.Cast(label>0, sitk.sitkUInt8)
    synthetic_label = sitk.Cast(synthetic_label>0, sitk.sitkUInt8)
    
    hausdorff_filter = sitk.HausdorffDistanceImageFilter()
    hausdorff_filter.Execute(label, synthetic_label)
    
    hd = hausdorff_filter.GetHausdorffDistance()
    
    return hd

In [4]:

class ContourInPlaneAug(object):
    
    def __init__(self, w_stdMM,h_stdMM, angle, ob_type="random"): #w_aMM, w_bMM are the measurements in MM
        
        self.w_stdMM = w_stdMM
        self.h_stdMM = h_stdMM
        self.angle = angle
        
        self.ob_type = ob_type
    
    def __call__(self, mask):  
        
        try:
     
            origin = mask.GetOrigin()
            spacing = mask.GetSpacing()
            direction = mask.GetDirection()

            w_spacing, h_spacing,_ = spacing

            mask_arr = sitk.GetArrayFromImage(mask)

            sys_type = np.random.choice(["inc","dec"])#behavior of the annotator

            out_mask = np.zeros_like(mask_arr)

            z_indeces = [i for i,mask_slice in enumerate(mask_arr) if mask_slice.sum()>0]

            for z in z_indeces:

                w_stdVOX = 0.0
                h_stdVOX = 0.0
                
                if self.ob_type=="random":
                    
                    w_stdVOX = np.ceil(np.random.uniform(-self.w_stdMM,self.w_stdMM)/w_spacing)
                    h_stdVOX = np.ceil(np.random.uniform(-self.h_stdMM,self.h_stdMM)/h_spacing)


                elif self.ob_type=="systematic":

                    if sys_type == "inc":
                        w_stdVOX = np.ceil(np.random.uniform(0,self.w_stdMM)/w_spacing)
                        h_stdVOX = np.ceil(np.random.uniform(0,self.h_stdMM)/h_spacing)
                    else:
                        w_stdVOX = np.ceil(np.random.uniform(-self.w_stdMM,0)/w_spacing)
                        h_stdVOX = np.ceil(np.random.uniform(-self.h_stdMM,0)/h_spacing)
                        
                

                props = measure.regionprops(mask_arr[z])
                w_min,h_min,w_max,h_max = props[0].bbox

                dw = w_max - w_min
                dh = h_max - h_min

                aug_dw = dw + w_stdVOX 
                aug_dh = dh + h_stdVOX 

                factor_w  = np.round(aug_dw/dw,2)
                factor_h = np.round(aug_dh/dh,2)

                if factor_w<=0:
                    continue;#donot execute further - no need to augment

                if factor_h<=0:
                    continue;

                mask_slice = sitk.GetImageFromArray(mask_arr[z])

                scales = (factor_w,factor_w,factor_h,factor_h, 1, 1)
                degrees = (0, 0, 0, 0, -self.angle, self.angle)
                transform = tio.RandomAffine(scales=scales,degrees=degrees,image_interpolation='nearest',p=1)

                aug_mask_slice = transform(mask_slice)
                out_mask[z] = sitk.GetArrayFromImage(aug_mask_slice)

            out_mask = sitk.GetImageFromArray(out_mask)
            out_mask.SetOrigin(origin)
            out_mask.SetSpacing(spacing)
            out_mask.SetDirection(direction)
        except Exception as e:
            print("Error with InPlane Augmentation",e)
            out_mask = mask
        
 
        return out_mask
    


In [5]:
class ContourOutPlaneAug(object):
    
    def __init__(self, scale_a, scale_b,angle, delta_z):
        
        self.scale_a = scale_a
        self.scale_b = scale_b
        
        self.delta_z = delta_z
        self.angle = angle
        
    def __call__(self, mask):
        
        try:
        
            origin = mask.GetOrigin()
            spacing = mask.GetSpacing()
            direction = mask.GetDirection()

            mask_arr = sitk.GetArrayFromImage(mask)

            aug_num_slices = np.random.randint(0, self.delta_z+1) #low, high (excluded high)

            z_indeces = [i for i,mask_slice in enumerate(mask_arr) if mask_slice.sum()>0]

            z_min, z_max = min(z_indeces), max(z_indeces)

            for i in range(aug_num_slices):

                dz = z_max-z_min

                if dz>0:

                    ref_z = np.random.choice([z_min,z_max])

                    offset = -1 if ref_z==z_min else 1

                    aug_type = "del" if np.random.uniform()>0.5 else "add"

                    if mask_arr[ref_z].sum()>0: #which means contour exists in that place, possible that it got deleted during iteration

                        if aug_type=="del":

                            if ref_z+offset in range(len(mask_arr)):

                                if mask_arr[ref_z+offset].sum()==0:#to check if new contours were already inserted up or down
                                    mask_arr[ref_z] = np.zeros(mask_arr[ref_z].shape)

                            else:#if ref_z+offset is outside the boundary
                                mask_arr[ref_z] = np.zeros(mask_arr[ref_z].shape)

                        elif aug_type=="add":

                            if ref_z+offset in range(len(mask_arr)):

                                if mask_arr[ref_z+offset].sum()==0:

                                    scales =(self.scale_a,self.scale_b, self.scale_a,self.scale_b, 1,1)
                                    degrees = (0,0, 0,0, -self.angle, self.angle)

                                    transform = tio.RandomAffine(scales=scales,degrees=degrees,image_interpolation='nearest',p=1)

                                    mask_slice = sitk.GetImageFromArray(mask_arr[ref_z])
                                    mask_arr[ref_z+offset] = sitk.GetArrayFromImage(transform(mask_slice))

            mask = sitk.GetImageFromArray(mask_arr)
            mask.SetOrigin(origin)
            mask.SetSpacing(spacing)
            mask.SetDirection(direction)
        
        except Exception as e:
            print("Error with OutPlane Augmentation",e)
      
        return mask



In [6]:
def extract_features(pid):
    
    featureRows = []
    
    img = sitk.ReadImage(os.path.join(DATA_DIR, pid, "CT.nii.gz"))

    for roi in ROI:
        
        mask = sitk.ReadImage(os.path.join(DATA_DIR, pid, f"{roi}.nii.gz"))

        ContourAug = tio.Compose([
            ContourInPlaneAug(**IN_AUG_PARAMS),
            ContourOutPlaneAug(**OUT_AUG_PARAMS)
        ])
        
        if ContourAug:
            
            for i in range(AUG_COUNT):
                
                aug_mask = None
                
                while True:

                    aug_mask = ContourAug(mask)
    
                    dice = compute_dice(mask, aug_mask)
                    hd = compute_hausdorff(mask, aug_mask)
                    
                    if dice>DICE_MIN and hd<HD_MIN:
                        break;
    
                extractor = radiomics.featureextractor.RadiomicsFeatureExtractor(PARAM_settingsFile,verbosity=False)

                featureVector = extractor.execute(img,aug_mask)

                featureVector['id'] = pid
                featureVector['roi'] = roi
                featureVector['dice'] = dice
                featureVector['hd'] = hd
                featureVector['judge'] = i+1

                featureRows.append(featureVector)

    return featureRows



In [7]:
# rows = extract_features(PIDS[0])

In [ ]:
df_rows = []
with ProcessPoolExecutor(NUM_WORKERS) as e:
    futures = [e.submit(extract_features, pid) for pid in PIDS]
    for f in tqdm(as_completed(futures), position=0, total = len(futures), desc="extracting features from perturbed masks"):
        rows = f.result()
        df_rows.append(rows)
        
pd.DataFrame(df_rows).to_parquet("results/perturbed_df.parquet")

extracting features from perturbed masks:   0%|          | 0/56 [00:00<?, ?it/s]

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Avera